# photoguard-removal on Colab

Before you start: **Runtime → Change runtime type → T4 GPU**.

Put your images in Google Drive at `MyDrive/photoguard-data/images/`, with `MyDrive/photoguard-data/SOURCES.csv` listing every file. Results go to `MyDrive/photoguard-results/`, so they survive a disconnect.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Replace `YOUR-USERNAME` with your GitHub username.

In [ ]:
!git clone https://github.com/YOUR-USERNAME/photoguard-removal.git
%cd photoguard-removal
!pip install -q -e ".[gpu,metrics,dev]"

## 2. Run the CPU tests
The same tests as on your laptop. Unimplemented tasks show as `xfailed`.

In [ ]:
!pytest -q

## 3. Bring in the data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/drive/MyDrive/photoguard-data/* data/
!ls data/images | head; wc -l data/SOURCES.csv

## 4. Smoke run
Two images and 50 PGD iterations. Note the seconds per image it prints, then size the full run to fit your session.

Attacks you haven't implemented yet are skipped, not crashed on.

In [ ]:
!python -m pgremoval.run --data data --out /content/drive/MyDrive/photoguard-results/smoke --limit 2 --iters 50

In [ ]:
!python -m pgremoval.summarise /content/drive/MyDrive/photoguard-results/smoke/metrics.csv

## 5. Look at the images
Always look before trusting the numbers. Does the protected edit look broken? Does it recover after each attack?

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

root = Path("/content/drive/MyDrive/photoguard-results/smoke/images")
img_dir = sorted(root.iterdir())[0]
names = ["clean", "protected", "edit_clean", "none_edit"] + sorted(
    p.stem for p in img_dir.glob("*_edit.png") if p.stem != "none_edit")
fig, axes = plt.subplots(1, len(names), figsize=(3 * len(names), 3))
for ax, n in zip(axes, names):
    ax.imshow(Image.open(img_dir / f"{n}.png")); ax.set_title(n, fontsize=9); ax.axis("off")
plt.tight_layout()

## 6. Full run
Resumable: if Colab disconnects, run this cell again and it carries on where it stopped.

In [ ]:
!python -m pgremoval.run --data data --out /content/drive/MyDrive/photoguard-results/full

In [ ]:
!python -m pgremoval.summarise /content/drive/MyDrive/photoguard-results/full/metrics.csv